# 1장 2강: 좋은 지표의 조건 — 실습문제

## 실습 목표

- Ravenstack의 비즈니스 목표에 맞춰 KPI, 보조 지표, 허무 지표를 구분할 수 있다.
- 구독 데이터에서 현재 반복 매출과 이탈률을 계산할 수 있다.
- 요금제 또는 유입 경로별 지표를 비교하여 개선이 필요한 대상을 찾을 수 있다.
- 행동 가능성, 비교 가능성, 이해 용이성을 기준으로 지표의 적절성을 판단할 수 있다.

## 실습 환경 / 데이터

- Python
- pandas
- `ravenstack_accounts.csv`
- `ravenstack_subscriptions.csv`

Ravenstack은 기업 고객에게 구독형 소프트웨어를 제공하는 B2B SaaS 서비스입니다.

이번 실습에서는 Ravenstack의 핵심 목표를 다음과 같이 가정합니다.

> **유료 구독을 안정적으로 유지하면서 반복 매출을 늘린다.**

주요 컬럼은 다음과 같습니다.

| 테이블 | 컬럼 | 의미 |
|---|---|---|
| accounts | `account_id` | 고객사 식별자 |
| accounts | `referral_source` | 고객사가 유입된 경로 |
| accounts | `signup_date` | 고객사 가입일 |
| accounts | `churn_flag` | 고객사 이탈 여부 |
| subscriptions | `subscription_id` | 구독 식별자 |
| subscriptions | `plan_tier` | 구독 요금제 |
| subscriptions | `mrr_amount` | 월간 반복 매출(MRR) |
| subscriptions | `is_trial` | 체험 구독 여부 |
| subscriptions | `end_date` | 구독 종료일 |
| subscriptions | `churn_flag` | 구독 이탈 여부 |

> 비율은 별도 지시가 없으면 소수점 둘째 자리의 백분율로 출력합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. 두 CSV 파일을 각각 `accounts`, `subscriptions`에 불러오세요.
3. 각 데이터의 행과 열 개수, 상위 5개 행을 확인하세요.
4. 분석 대상 컬럼의 자료형과 결측치 개수를 확인하세요.
5. `signup_date`, `start_date`, `end_date`를 날짜형으로 변환하세요.
6. `mrr_amount`의 기초 통계량과 `plan_tier`의 빈도를 확인하세요.

In [14]:
# 실습 준비 코드를 작성하세요.

# 1. 필요한 라이브러리 불러오기
import pandas as pd


# 2. 두 CSV 파일 불러오기
accounts = pd.read_csv("ravenstack_accounts.csv")
subscriptions = pd.read_csv("ravenstack_subscriptions.csv")


# 3. 각 데이터의 행과 열 개수, 상위 5개 행 확인
print("=== accounts ===")
print("행 개수:", accounts.shape[0])
print("열 개수:", accounts.shape[1])
display(accounts.head())

print("\n=== subscriptions ===")
print("행 개수:", subscriptions.shape[0])
print("열 개수:", subscriptions.shape[1])
display(subscriptions.head())


# 4. 분석 대상 컬럼의 자료형과 결측치 개수 확인
print("=== accounts 자료형 ===")
print(accounts[[
    "account_id",
    "referral_source",
    "signup_date",
    "churn_flag"
]].dtypes)

print("\n=== accounts 결측치 개수 ===")
print(accounts[[
    "account_id",
    "referral_source",
    "signup_date",
    "churn_flag"
]].isnull().sum())


print("\n=== subscriptions 자료형 ===")
print(subscriptions[[
    "subscription_id",
    "account_id",
    "plan_tier",
    "mrr_amount",
    "is_trial",
    "start_date",
    "end_date",
    "churn_flag"
]].dtypes)

print("\n=== subscriptions 결측치 개수 ===")
print(subscriptions[[
    "subscription_id",
    "account_id",
    "plan_tier",
    "mrr_amount",
    "is_trial",
    "start_date",
    "end_date",
    "churn_flag"
]].isnull().sum())


# 5. 날짜형으로 변환
accounts["signup_date"] = pd.to_datetime(accounts["signup_date"])

subscriptions["start_date"] = pd.to_datetime(
    subscriptions["start_date"]
)

subscriptions["end_date"] = pd.to_datetime(
    subscriptions["end_date"]
)


# 날짜형 변환 확인
print("\n=== 날짜형 변환 확인 ===")
print(accounts["signup_date"].dtype)
print(subscriptions["start_date"].dtype)
print(subscriptions["end_date"].dtype)


# 6. mrr_amount 기초 통계량과 plan_tier 빈도 확인
print("\n=== mrr_amount 기초 통계량 ===")
display(subscriptions["mrr_amount"].describe())


print("\n=== plan_tier 빈도 ===")
print(subscriptions["plan_tier"].value_counts())


=== accounts ===
행 개수: 500
열 개수: 10


,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True



=== subscriptions ===
행 개수: 5000
열 개수: 14


,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,False,False,False,True,monthly,True
1,S-0f6f44,A-9b9fe9,2024-06-11,NaN,Pro,17,833,9996,False,False,False,False,monthly,True
2,S-51c0d1,A-659280,2024-11-25,NaN,Enterprise,62,0,0,True,True,False,False,annual,False
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,False,False,False,True,monthly,True
4,S-cff5a2,A-ba6516,2024-01-10,NaN,Enterprise,27,5373,64476,False,False,False,False,monthly,True


=== accounts 자료형 ===
account_id          str
referral_source     str
signup_date         str
churn_flag         bool
dtype: object

=== accounts 결측치 개수 ===
account_id         0
referral_source    0
signup_date        0
churn_flag         0
dtype: int64

=== subscriptions 자료형 ===
subscription_id      str
account_id           str
plan_tier            str
mrr_amount         int64
is_trial            bool
start_date           str
end_date             str
churn_flag          bool
dtype: object

=== subscriptions 결측치 개수 ===
subscription_id       0
account_id            0
plan_tier             0
mrr_amount            0
is_trial              0
start_date            0
end_date           4514
churn_flag            0
dtype: int64

=== 날짜형 변환 확인 ===
datetime64[us]
datetime64[us]
datetime64[us]

=== mrr_amount 기초 통계량 ===


count     5000.000000
mean      2267.749400
std       3421.375348
min          0.000000
25%        285.000000
50%        931.000000
75%       2786.000000
max      33830.000000
Name: mrr_amount, dtype: float64


=== plan_tier 빈도 ===
plan_tier
Enterprise    1723
Pro           1675
Basic         1602
Name: count, dtype: int64


---

## 필수 1. 비즈니스 목표에 맞는 지표 구분하기

### 문제 1-1. 현재 반복 매출을 중심으로 지표를 계산하고 분류하기

#### 문제 설명

Ravenstack은 **유료 구독을 안정적으로 유지하면서 반복 매출을 늘리는 것**을 핵심 목표로 정했습니다.

다음 세 지표를 직접 계산한 뒤 현재 목표를 기준으로 KPI, 보조 지표, 허무 지표로 분류하세요.

- 현재 유료 구독 MRR
- 전체 구독 이탈률
- 누적 가입 고객사 수

이번 실습에서는 `end_date`가 비어 있고 `is_trial`이 `False`인 구독을 **현재 활성 유료 구독**으로 정의합니다.

#### 요구사항

1. 현재 활성 유료 구독 조건을 불리언 변수 `active_paid_mask`로 만드세요.
2. 현재 활성 유료 구독의 `mrr_amount` 합계를 `current_paid_mrr`로 계산하세요.
3. `subscriptions`의 `churn_flag` 평균으로 `subscription_churn_rate`를 계산하세요.
4. `accounts`의 고유한 `account_id` 개수를 `cumulative_accounts`로 계산하세요.
5. 세 지표를 알아보기 쉬운 형식으로 출력하세요.
6. 현재 비즈니스 목표를 기준으로 다음과 같이 분류하고 이유를 설명하세요.
   - 현재 유료 구독 MRR: KPI
   - 전체 구독 이탈률: 보조 지표
   - 누적 가입 고객사 수: 허무 지표 후보

#### 해석 질문

**Q1.** 현재 유료 구독 MRR을 KPI로 볼 수 있는 이유는 무엇인가요?  
**Q2.** 전체 구독 이탈률은 현재 유료 구독 MRR을 이해하는 데 어떻게 도움을 주나요?  
**Q3.** 누적 가입 고객사 수만으로 현재 서비스가 성장한다고 판단하기 어려운 이유는 무엇인가요?

#### 제출 결과

- 세 지표의 계산 코드와 결과
- KPI, 보조 지표, 허무 지표 분류
- 분류 이유
- Q1~Q3 답변

In [ ]:
# 필수 1 코드를 작성하세요.

# 1. 현재 활성 유료 구독 조건 만들기
active_paid_mask = (subscriptions["end_date"].isna()& (subscriptions["is_trial"] == False))
# 2. 현재 활성 유료 구독의 MRR 합계 계산
current_paid_mrr = subscriptions.loc[active_paid_mask, "mrr_amount"].sum()
# 3. 전체 구독 이탈률 계산
subscription_churn_rate = subscriptions["churn_flag"].mean() * 100
# 4. 누적 가입 고객사 수 계산
cumulative_accounts = accounts["account_id"].nunique()


# 5. 세 지표 출력
print("=== Ravenstack 주요 지표 ===")
print(f"현재 유료 구독 MRR: ${current_paid_mrr:,.2f}")
print(f"전체 구독 이탈률: {subscription_churn_rate:.2f}%")
print(f"누적 가입 고객사 수: {cumulative_accounts:,}개")



=== Ravenstack 주요 지표 ===
현재 유료 구독 MRR: $10,159,608.00
전체 구독 이탈률: 9.72%
누적 가입 고객사 수: 500개


### 필수 1 답변 작성란
 
- **Q1.**  
현재 활성 유료 구독이 매 달 만들어내는 반복 매출을 직접 나타내므로 핵심 목표와 일치  

- **Q2.**  
이탈율이 높아지면 유료 구독과 반복 매출이 감소할 가능성이 있습니다  

- **Q3.**  
누적 가입 고객사 수는 이미 이탈했거나 구독을 실제로 진행하지 않는 고객사들도 포함될 수 있다



---

## 필수 2. 요금제별 핵심 지표 비교하기

### 문제 2-1. 어느 요금제를 우선적으로 살펴봐야 하는가?

#### 문제 설명

전체 평균만 확인하면 요금제별 차이를 놓칠 수 있습니다. `Basic`, `Pro`, `Enterprise` 요금제별로 구독 규모, 이탈률, 현재 유료 구독 MRR을 비교하세요.

#### 요구사항

1. `subscriptions`에 `is_active_paid` 컬럼을 만드세요.
   - `end_date`가 비어 있고 `is_trial`이 `False`이면 `True`
2. `plan_tier`별로 다음 값을 집계하여 `plan_metrics`를 만드세요.
   - 전체 구독 수
   - 이탈 구독 수
   - 구독 이탈률
   - 현재 활성 유료 구독 수
3. 현재 활성 유료 구독만 사용해 요금제별 MRR 합계를 계산하고 `active_mrr` 컬럼으로 추가하세요.
4. 이탈률은 백분율로, MRR은 천 단위 구분 기호를 사용하여 출력하세요.
5. 현재 유료 구독 MRR이 가장 큰 요금제와 이탈률이 가장 높은 요금제를 확인하세요.
6. 현재 목표를 고려하여 우선적으로 점검할 요금제 하나를 정하고, 데이터 근거와 확인할 개선 방향을 설명하세요.

#### 해석 질문

**Q1.** 요금제별 이탈률을 비교하는 것이 전체 이탈률만 확인하는 것보다 행동 가능성이 높은 이유는 무엇인가요?  
**Q2.** 현재 유료 구독 MRR이 가장 큰 요금제는 무엇인가요?  
**Q3.** 이탈률이 가장 높은 요금제는 무엇인가요?  
**Q4.** 위 두 결과를 함께 보면 어떤 요금제를 우선적으로 점검할 수 있으며, 그 이유는 무엇인가요?

#### 제출 결과

- `plan_metrics` 집계 코드와 결과
- MRR 및 이탈률 기준 요금제 비교
- 우선 점검 대상과 개선 방향
- Q1~Q4 답변

In [13]:
# 필수 2 코드를 작성하세요.

# 1. 현재 활성 유료 구독 여부 컬럼 생성
subscriptions["is_active_paid"] = (
    subscriptions["end_date"].isna()
    & (subscriptions["is_trial"] == False)
)


# 2. 요금제별 핵심 지표 집계
plan_metrics = (
    subscriptions
    .groupby("plan_tier")
    .agg(
        total_subscriptions=("subscription_id", "count"),
        churned_subscriptions=("churn_flag", "sum"),
        churn_rate=("churn_flag", "mean"),
        active_paid_subscriptions=("is_active_paid", "sum")
    )
    .reset_index()
)


# 3. 현재 활성 유료 구독의 요금제별 MRR 계산
active_mrr = (
    subscriptions[subscriptions["is_active_paid"]]
    .groupby("plan_tier")["mrr_amount"]
    .sum()
    .reset_index()
    .rename(columns={"mrr_amount": "active_mrr"})
)


# plan_metrics에 MRR 추가
plan_metrics = plan_metrics.merge(
    active_mrr,
    on="plan_tier",
    how="left"
)


# 4. 이탈률 백분율로 변환
plan_metrics["churn_rate"] = (
    plan_metrics["churn_rate"] * 100
)


# 결과 확인
display(plan_metrics)

,plan_tier,total_subscriptions,churned_subscriptions,churn_rate,active_paid_subscriptions,active_mrr
0,Basic,1602,152,9.488140,1228,687914
1,Enterprise,1723,172,9.982589,1304,7546876
2,Pro,1675,162,9.671642,1282,1924818


### 필수 2 답변 작성란

- **Q1.**
 전체 이탈율은 서비스 전체의 현상을 보여주고 요금제별 이탈율은 어느 요금제를 먼저 점검해야하는지를 판단한다  
 다음 회의가 열린다면 담당 팀이 개선 대상을 명확하게 설정하여 개선점을 진행할 수 있다  

- **Q2.**  

- **Q3.**  

- **Q4.**  


---

## 과제 1. 평가 문항 기반 독립 과제

### 문제 3-1. 유입 경로별 고객사 이탈률 비교하기

#### 문제 설명

Ravenstack 마케팅팀은 고객사가 유입된 경로에 따라 이탈 수준이 다른지 확인하려고 합니다. 유입 경로별 고객사 수와 이탈률을 비교하여 우선 점검할 유입 경로를 찾으세요.

> 이 과제는 필수 2에서 수행한 그룹별 지표 집계와 해석을 새로운 컬럼에 동일하게 적용하는 문제입니다.

#### 요구사항

1. `accounts`를 `referral_source`별로 그룹화하세요.
2. 다음 값을 집계하여 `source_metrics`를 만드세요.
   - 전체 고객사 수
   - 이탈 고객사 수
   - 고객사 이탈률
3. 고객사 이탈률이 높은 순서로 정렬하세요.
4. 이탈률은 소수점 둘째 자리의 백분율로 출력하세요.
5. 이탈률이 가장 높은 유입 경로를 찾으세요.
6. 해당 유입 경로를 우선 점검 대상으로 정하고, 확인할 개선 방향을 한 가지 제안하세요.
7. 유입 경로별 이탈률을 좋은 지표의 세 조건으로 평가하세요.
   - 행동 가능성
   - 비교 가능성
   - 이해 용이성

#### 해석 질문

**Q1.** 고객사 이탈률이 가장 높은 유입 경로는 무엇인가요?  
**Q2.** 유입 경로별 이탈률은 마케팅팀의 행동으로 어떻게 연결할 수 있나요?  
**Q3.** 유입 경로별 이탈률은 좋은 지표의 세 조건을 충족하나요?
**Q4.** KPI·보조 지표·허무 지표는 어떻게 다른가요? 고객사 이탈을 줄이려는 목적에서 각각의 지표 예시와 선정 이유를 제시하세요. 허무 지표는 어떤 맥락에서 성과 판단에 도움이 되지 않는지도 설명하세요.

#### 제출 결과

- `source_metrics` 집계 코드와 결과
- 우선 점검할 유입 경로
- 개선 방향
- 좋은 지표의 조건 평가
- Q1~Q4 답변

In [ ]:
# 과제 1 코드를 작성하세요.

### 과제 1 답변 작성란

- **Q1.**
- **Q2.**
- **Q3.**
- **Q4.**

---

## 실습 마무리

아래 질문에 답하세요.

1. 이번 실습에서 해결하려고 한 비즈니스 문제는 무엇인가요?  
- 유료 구독을 안정적으로 유지하면서 반복 매출을 늘리기 위한 핵심 지표를 관리하고  
- 어느 대상을 우선적으로 개선해야하는지 확인했다  

2. 현재 목표를 직접 보여주는 KPI로 어떤 지표를 선택했나요?  
- 현재 활성화된 유료 구독자들이 만드는 매출을 MRR을 선택했다  

3. KPI의 변화를 이해하기 위해 어떤 보조 지표를 확인했나요?  
- 전체 구독 이탈율과 요금제별 이탈율을 확인했다  
- 이탈율은 현재 또는 미래의 매출 감소로 이어지기 때문이다  


4. 허무 지표 후보를 핵심 성과로 사용할 때 어떤 문제가 생길 수 있나요?  
- 누적 가입 고객사를 핵심 성과로 사용한다면 현재 구독을 하지 않거나 활동하지 않아도  
실제 성과를 과대평가할 수가 있다  

5. 어떤 분석 결과를 근거로 개선 대상을 정했나요?  
요금제별로 MRR과 이탈율을 분석하여 EnterPrise를 우선 점검 대상으로 정했다